# 📡 RF Modulation Classification (RFMC) — RadioML 2016.10A Model Training

This notebook trains a deep learning RF modulation classifier (`RFNet` / `RFCLDNN` / `RFTransformer`) on the **RadioML 2016.10A** dataset using Google Colab's GPU acceleration.

### Pipeline Overview:
1. **Environment Setup**: Verify GPU acceleration (`T4` / `V100` / `A100`).
2. **Dataset Setup**: Load `rml2016.npz` (or extract and convert `RML2016.10a.tar.bz2`).
3. **Model Training**: Train with stratified splits, SNR-weighted loss, channel impairments augmentation, and cosine LR scheduling.
4. **Evaluation**: Generate Accuracy vs. SNR curves and normalized Confusion Matrices.
5. **Open-Set Calibration**: Calibrate MSP, Energy, and Mahalanobis unknown rejection thresholds at 95% target TPR.
6. **Export**: Package weights (`rfnet_best.pt`), `model_card.json`, and acceptance files for deployment.

> **Tip:** Ensure GPU runtime is active: Go to `Runtime` → `Change runtime type` → Select `T4 GPU`.

In [ ]:
# 1. Hardware & CUDA Check
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## 2. Setup Code & Repository

In [ ]:
import os
from pathlib import Path

# If you are cloning from GitHub, uncomment and set your repository URL:
# !git clone https://github.com/<your-username>/rf-mod-classifier.git
# %cd rf-mod-classifier

# Alternatively, if you uploaded the project zip or working directory directly:
print("Working directory:", os.getcwd())
!ls -lh

# Install dependencies
!pip install -q scikit-learn matplotlib

## 3. Prepare the RadioML 2016.10A Dataset

Choose one of the following methods to provide the dataset in Colab:
- **Option A**: Upload `data/rml2016.npz` directly from your local machine (recommended, already power-normalised, ~207 MB).
- **Option B**: Mount Google Drive where you saved `rml2016.npz` or `RML2016.10a.tar.bz2`.
- **Option C**: Upload or download `RML2016.10a.tar.bz2` and run conversion.

In [ ]:
!mkdir -p data models eval_out

# OPTION A: Upload rml2016.npz directly from your local computer
# from google.colab import files
# print("Select and upload data/rml2016.npz:")
# uploaded = files.upload()
# for fname in uploaded.keys():
#     !mv "{fname}" data/

# OPTION B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/rml2016.npz data/

# If you have the raw RML2016.10a.tar.bz2 file in data/:
if os.path.exists("data/RML2016.10a.tar.bz2") and not os.path.exists("data/rml2016.npz"):
    print("Extracting RML2016.10a.tar.bz2...")
    !tar -xjf data/RML2016.10a.tar.bz2 -C data/
    pkl = "data/RML2016.10a_dict.pkl"
    if not os.path.exists(pkl):
        pkl = "data/RML2016.10a_dict_optimized.pkl"
    print(f"Converting {pkl} to data/rml2016.npz ...")
    !python scripts/convert_radioml.py rml2016 "{pkl}" -o data/rml2016.npz

In [ ]:
# Verify dataset integrity
import numpy as np
assert os.path.exists("data/rml2016.npz"), "data/rml2016.npz not found! Upload or place it in data/"
ds = np.load("data/rml2016.npz")
print("✅ Dataset loaded successfully:")
print(f"  Total samples : {len(ds['X']):,}")
print(f"  Signal shape  : {ds['X'].shape[1:]} (2 I/Q channels × {ds['X'].shape[2]} samples)")
print(f"  Classes ({len(ds['class_names'])}): {list(ds['class_names'])}")
print(f"  SNR span      : [{ds['snr'].min()}, {ds['snr'].max()}] dB")

## 4. Train Model with GPU Acceleration

Choose your backbone architecture:
- `cldnn`: CNN + BiLSTM (recommended for low-to-mid SNR performance).
- `rfnet`: 4-block Conv1D residual-style baseline (fast, compact).
- `transformer`: 1D Self-Attention Encoder.

We use `--snr-weighted` so that lower-SNR samples have proportional emphasis in loss, preventing high-SNR dominance.

In [ ]:
!python scripts/train.py \
    --data data/rml2016.npz \
    --arch cldnn \
    --snr-weighted \
    --epochs 60 \
    --batch-size 256 \
    --lr 0.001 \
    --lr-schedule cosine \
    --warmup-epochs 3 \
    --input-length 128 \
    --device cuda \
    --out models/

## 5. Standalone Evaluation & Production Acceptance Gate Export

In [ ]:
!python scripts/evaluate.py \
    --model models/rfnet_best.pt \
    --model-card models/model_card.json \
    --data data/rml2016.npz \
    --acceptance \
    --out eval_out/

## 6. Open-Set Calibration on Held-Out Unknown Waveforms

Calibrate detection boundaries at 95% target True Positive Rate (TPR) using held-out unfamiliar modulations (`OOK`, `GMSK`, `OFDM_QPSK`, `CHIRP`, `NOISE`).

In [ ]:
!python scripts/calibrate_openset.py \
    --model models/rfnet_best.pt \
    --model-card models/model_card.json \
    --simulate \
    --tpr 0.95 \
    --out models/openset

## 7. Results & Visualizations

In [ ]:
from IPython.display import Image, display
import json

# Display Model Card Summary
if os.path.exists("models/model_card.json"):
    with open("models/model_card.json") as f:
        card = json.load(f)
    print("=" * 60)
    print(f"Model Architecture: {card['model']}")
    print(f"Total Parameters  : {card['parameters']:,}")
    print(f"Test Accuracy     : {card['results']['test_accuracy']:.2%}")
    print(f"High SNR (>=10dB) : {card['results']['accuracy_high_snr_gte_10dB']:.2%}")
    print(f"Mid SNR (~0dB)    : {card['results']['accuracy_mid_snr_0dB']:.2%}")
    print(f"Low SNR (<=-4dB)  : {card['results']['accuracy_low_snr_lte_neg4dB']:.2%}")
    print(f"SHA-256 Hash      : {card['sha256']}")
    print("=" * 60)

# Display Plots
if os.path.exists("models/accuracy_vs_snr.png"):
    print("▶ Accuracy vs. SNR:")
    display(Image("models/accuracy_vs_snr.png"))

if os.path.exists("models/confusion_matrix.png"):
    print("▶ Confusion Matrix:")
    display(Image("models/confusion_matrix.png"))

if os.path.exists("models/openset/auroc_vs_snr.png"):
    print("▶ Open-Set AUROC vs. SNR:")
    display(Image("models/openset/auroc_vs_snr.png"))

## 8. Download Trained Artifacts to Local Computer

In [ ]:
from google.colab import files

# Create zip archive of models and evaluation results
!zip -r rfmc_trained_rml2016.zip models/ eval_out/

# Download to your local machine
files.download("rfmc_trained_rml2016.zip")
print("✅ Zip file ready and download triggered!")